<a href="https://colab.research.google.com/github/Swas26/NLP-ESG/blob/main/NLP_Assessment_3_Filtering_and_Scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClimateBERT Filtering & Scoring
**Pipeline:** combined.csv → climate-relevance filter → sentiment scoring → outputs

**Loads data directly from GitHub** — no need to re-run preprocessing.

**Models used:**
- `climatebert/distilroberta-base-climate-detector` — filters out non-climate text
- `climatebert/distilroberta-base-climate-sentiment` — scores remaining text positive/negative/neutral

**Outputs pushed to `data/processed/` on GitHub:**
- `filtered.csv` — climate-relevant rows only
- `individual_scores.csv` — per-row sentiment scores
- `company_summary.csv` — average sentiment score per company (greenwashing signal)

In [ ]:
# --- SETUP ---
# Pulls latest processed data from GitHub — no Drive mounting, no reprocessing
# GITHUB_TOKEN must be set in Colab Secrets (🔑 icon in sidebar)
# Generate one at: github.com/settings/tokens (scope: repo)

!pip install transformers torch -q

import os
import pandas as pd
import torch
from google.colab import userdata

OUTPUT_DIR = '/content/outputs'
REPO       = 'Swas26/NLP-ESG'
REPO_DIR   = '/content/repo'
GIT_EMAIL  = 'aatman.dwivedi@student.uts.edu.au'
GIT_NAME   = 'Swas26'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Pull latest data from GitHub
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git {REPO_DIR} -q

combined_path = f'{REPO_DIR}/data/processed/combined.csv'
if not os.path.exists(combined_path):
    raise FileNotFoundError("combined.csv not found in repo — run data_preprocessing.ipynb first")

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# --- LOAD DATA ---

df = pd.read_csv(combined_path)
print(f"Loaded {len(df)} rows from GitHub")
print(df['source_type'].value_counts())

In [ ]:
# --- PART 1: CLIMATE RELEVANCE FILTERING ---
# ClimateBERT detector classifies each chunk as climate-related or not
# Keeps only rows where label == 'yes' to reduce noise before scoring
# Text truncated to 1500 chars to stay within model token limits

from transformers import pipeline

detector = pipeline(
    'text-classification',
    model='climatebert/distilroberta-base-climate-detector'
)

def is_climate_related(text):
    text = str(text)[:1500]
    try:
        result = detector(text)[0]
        return result['label'] == 'yes'
    except:
        return False

print("Filtering with ClimateBERT detector...")
df['is_relevant'] = df['text'].apply(is_climate_related)

before = len(df)
filtered_df = df[df['is_relevant'] == True].drop(columns=['is_relevant'])
after = len(filtered_df)

print(f"Kept {after} / {before} rows ({round(after/before*100)}% climate-relevant)")
print(filtered_df['source_type'].value_counts())

filtered_df.to_csv(f'{OUTPUT_DIR}/filtered.csv', index=False)

In [ ]:
# --- PART 2: SENTIMENT SCORING ---
# ClimateBERT sentiment model outputs positive/negative/neutral probabilities
# Score = P(positive) - P(negative): higher = more positive climate language = greenwashing risk

sentiment = pipeline(
    'text-classification',
    model='climatebert/distilroberta-base-climate-sentiment',
    return_all_scores=True
)

def get_sentiment_score(text):
    text = str(text)[:1500]
    try:
        scores = sentiment(text)[0]  # list of {label, score} dicts
        score_dict = {item['label']: item['score'] for item in scores}
        p_pos = score_dict.get('positive', 0)
        p_neg = score_dict.get('negative', 0)
        return p_pos - p_neg  # net score: >0 positive lean, <0 negative lean
    except:
        return None

print("Running ClimateBERT sentiment scoring...")
filtered_df['sentiment_score'] = filtered_df['text'].apply(get_sentiment_score)

print(f"Scored {filtered_df['sentiment_score'].notna().sum()} / {len(filtered_df)} rows")
print(filtered_df['sentiment_score'].describe())

filtered_df.to_csv(f'{OUTPUT_DIR}/individual_scores.csv', index=False)

In [ ]:
# --- PART 3: COMPANY SUMMARY ---
# Aggregates sentiment scores per company
# Higher average = more positive climate language = potential greenwashing signal

results = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')

summary = results.groupby('company')['sentiment_score'].agg(['mean', 'count']).round(4)
summary.columns = ['avg_sentiment', 'num_chunks']
summary = summary.sort_values('avg_sentiment', ascending=False)

print(summary)
summary.to_csv(f'{OUTPUT_DIR}/company_summary.csv')

In [ ]:
# --- PUSH OUTPUTS TO GITHUB ---
# Copies scoring outputs into the repo and pushes to data/processed/
# Teammates just run `git pull` to get the latest results

import shutil

processed_dir = f'{REPO_DIR}/data/processed'
os.makedirs(processed_dir, exist_ok=True)

for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.csv'):
        shutil.copy(f'{OUTPUT_DIR}/{f}', f'{processed_dir}/{f}')
        print(f"Copied {f}")

!cd {REPO_DIR} && git config user.email "{GIT_EMAIL}"
!cd {REPO_DIR} && git config user.name "{GIT_NAME}"
!cd {REPO_DIR} && git add data/processed/
!cd {REPO_DIR} && git commit -m "update scoring outputs"
!cd {REPO_DIR} && git push

print("\nAll outputs pushed to GitHub → data/processed/")